In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

project_root = Path.home() / "Documents" / "airline-delay-reliability-analysis"

monthly_dir = project_root / "data" / "processed" / "monthly"

powerbi_dir = project_root / "data" / "powerbi"
powerbi_dir.mkdir(parents=True, exist_ok=True)

parquet_files = sorted(
    monthly_dir.glob("flights_2025_*.parquet")
)

print(f"Power BI output folder: {powerbi_dir}")
print(f"Monthly source files found: {len(parquet_files)}")

Power BI output folder: C:\Users\21129\Documents\airline-delay-reliability-analysis\data\powerbi
Monthly source files found: 12


In [2]:
dashboard_columns = [
    "FlightDate",
    "Month",
    "DayOfWeek",
    "Marketing_Airline_Network",
    "Origin",
    "Dest",
    "CRSDepTime",
    "ArrDelayMinutes",
    "ArrDel15",
    "Cancelled",
    "Diverted",
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay"
]

monthly_dashboard_dfs = []

for file in parquet_files:
    month_df = pd.read_parquet(
        file,
        columns=dashboard_columns,
        engine="pyarrow"
    )

    monthly_dashboard_dfs.append(month_df)

flight_dashboard = pd.concat(
    monthly_dashboard_dfs,
    ignore_index=True
)

flight_dashboard["FlightDate"] = pd.to_datetime(
    flight_dashboard["FlightDate"]
)

flight_dashboard["Route"] = (
    flight_dashboard["Origin"]
    + " → "
    + flight_dashboard["Dest"]
)

flight_dashboard["CompletedArrivalFlag"] = (
    flight_dashboard["ArrDel15"].notna().astype(int)
)

flight_dashboard["OnTimeArrivalFlag"] = (
    flight_dashboard["ArrDel15"].eq(0).astype(int)
)

flight_dashboard["DelayedArrivalFlag"] = (
    flight_dashboard["ArrDel15"].eq(1).astype(int)
)

flight_dashboard["ActionableDelayMinutes"] = (
    flight_dashboard["CarrierDelay"].fillna(0)
    + flight_dashboard["LateAircraftDelay"].fillna(0)
)

dep_hour = (
    flight_dashboard["CRSDepTime"]
    .fillna(0)
    .astype(int)
    // 100
)

dep_hour = dep_hour.replace(24, 0)

flight_dashboard["DepartureTimeBand"] = pd.cut(
    dep_hour,
    bins=[-1, 7, 10, 13, 16, 19, 23],
    labels=[
        "Early Morning",
        "Morning",
        "Midday",
        "Afternoon",
        "Evening",
        "Night"
    ]
)

output_path = (
    powerbi_dir
    / "flight_dashboard.parquet"
)

flight_dashboard.to_parquet(
    output_path,
    index=False,
    engine="pyarrow"
)

file_size_mb = (
    output_path.stat().st_size
    / 1024**2
)

print(f"Rows: {len(flight_dashboard):,}")
print(f"Columns: {flight_dashboard.shape[1]}")
print(f"File size: {file_size_mb:.2f} MB")
print(f"Saved to: {output_path}")

Rows: 4,518,993
Columns: 22
File size: 29.27 MB
Saved to: C:\Users\21129\Documents\airline-delay-reliability-analysis\data\powerbi\flight_dashboard.parquet


In [3]:
delay_cause_columns = [
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay"
]

core_airports = [
    "ATL", "DFW", "DEN", "ORD",
    "CLT", "MIA", "PHX",
    "DTW", "MSP", "SLC",
    "JFK", "LGA",
    "IAH", "EWR", "SFO", "LAX"
]

priority_airports = [
    "ORD", "DFW", "ATL",
    "DEN", "CLT", "LGA"
]

flight_dashboard["AttributedDelayMinutes"] = (
    flight_dashboard[delay_cause_columns]
    .fillna(0)
    .sum(axis=1)
)

# -----------------------------------
# 1. Airport priority table
# -----------------------------------

airport_priority = (
    flight_dashboard[
        flight_dashboard["Origin"].isin(core_airports)
    ]
    .groupby("Origin")
    .agg(
        ScheduledDepartures=("Origin", "size"),
        CompletedArrivals=("CompletedArrivalFlag", "sum"),
        DelayedArrivals=("DelayedArrivalFlag", "sum"),
        TotalArrivalDelayMinutes=("ArrDelayMinutes", "sum"),
        ActionableDelayMinutes=("ActionableDelayMinutes", "sum"),
        AttributedDelayMinutes=("AttributedDelayMinutes", "sum")
    )
    .reset_index()
)

airport_priority["DelayRatePct"] = (
    airport_priority["DelayedArrivals"]
    / airport_priority["CompletedArrivals"]
    * 100
)

airport_priority["ActionableDelayIntensity"] = (
    airport_priority["ActionableDelayMinutes"]
    / airport_priority["CompletedArrivals"]
    * 1000
)

airport_priority["ActionableSharePct"] = (
    airport_priority["ActionableDelayMinutes"]
    / airport_priority["AttributedDelayMinutes"]
    * 100
)

airport_priority["PriorityAirportFlag"] = (
    airport_priority["Origin"]
    .isin(priority_airports)
    .astype(int)
)

airport_priority = airport_priority.sort_values(
    "ActionableDelayMinutes",
    ascending=False
)

airport_priority_path = (
    powerbi_dir / "airport_priority.csv"
)

airport_priority.to_csv(
    airport_priority_path,
    index=False
)


# -----------------------------------
# 2. Route priority table
# -----------------------------------

route_priority = (
    flight_dashboard
    .groupby(["Origin", "Dest", "Route"])
    .agg(
        ScheduledFlights=("Route", "size"),
        CompletedArrivals=("CompletedArrivalFlag", "sum"),
        DelayedArrivals=("DelayedArrivalFlag", "sum"),
        TotalArrivalDelayMinutes=("ArrDelayMinutes", "sum"),
        ActionableDelayMinutes=("ActionableDelayMinutes", "sum"),
        AttributedDelayMinutes=("AttributedDelayMinutes", "sum")
    )
    .reset_index()
)

route_priority = route_priority[
    route_priority["ScheduledFlights"] >= 1000
].copy()

route_priority["DelayRatePct"] = (
    route_priority["DelayedArrivals"]
    / route_priority["CompletedArrivals"]
    * 100
)

route_priority["ActionableDelayIntensity"] = (
    route_priority["ActionableDelayMinutes"]
    / route_priority["CompletedArrivals"]
    * 1000
)

route_priority["ActionableSharePct"] = (
    route_priority["ActionableDelayMinutes"]
    / route_priority["AttributedDelayMinutes"]
    * 100
)

route_priority = route_priority.sort_values(
    "ActionableDelayMinutes",
    ascending=False
)

route_priority_path = (
    powerbi_dir / "route_priority.csv"
)

route_priority.to_csv(
    route_priority_path,
    index=False
)


# -----------------------------------
# 3. Financial scenario table
# -----------------------------------

targeted_baseline = (
    airport_priority.loc[
        airport_priority["Origin"].isin(priority_airports),
        "ActionableDelayMinutes"
    ]
    .sum()
)

scenario_rates = [
    ("5%", 0.05),
    ("10%", 0.10),
    ("15%", 0.15)
]

cost_assumptions = [
    ("Low", 20),
    ("Base", 40),
    ("High", 60)
]

financial_rows = []

for scenario_label, reduction_rate in scenario_rates:
    for cost_label, cost_per_minute in cost_assumptions:

        minutes_saved = (
            targeted_baseline
            * reduction_rate
        )

        financial_rows.append({
            "Scope": "Targeted 6 Airports",
            "ReductionScenario": scenario_label,
            "ReductionRate": reduction_rate,
            "CostScenario": cost_label,
            "CostPerMinute": cost_per_minute,
            "BaselineActionableDelayMinutes": targeted_baseline,
            "DelayMinutesSaved": minutes_saved,
            "EstimatedOperatingValueUSD": (
                minutes_saved * cost_per_minute
            ),
            "EstimatedOperatingValueMillions": (
                minutes_saved
                * cost_per_minute
                / 1_000_000
            )
        })

financial_scenarios = pd.DataFrame(
    financial_rows
)

financial_scenarios_path = (
    powerbi_dir / "financial_scenarios.csv"
)

financial_scenarios.to_csv(
    financial_scenarios_path,
    index=False
)


print("Power BI management tables created:")
print(f"Airport table rows: {len(airport_priority)}")
print(f"Route table rows: {len(route_priority)}")
print(f"Financial scenario rows: {len(financial_scenarios)}")

print()
print(airport_priority_path)
print(route_priority_path)
print(financial_scenarios_path)

Power BI management tables created:
Airport table rows: 16
Route table rows: 1785
Financial scenario rows: 9

C:\Users\21129\Documents\airline-delay-reliability-analysis\data\powerbi\airport_priority.csv
C:\Users\21129\Documents\airline-delay-reliability-analysis\data\powerbi\route_priority.csv
C:\Users\21129\Documents\airline-delay-reliability-analysis\data\powerbi\financial_scenarios.csv
